<a href="https://colab.research.google.com/github/nawroz-m/ML_learning/blob/main/Document_Processing_with_PaddleOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dcument Processing with PaddleOCR

In [ ]:
# Install all functionalities, such as document parsing, document understanding,
# document translation, and key information extraction
!pip install "paddleocr[all]"

In [ ]:
# You need to install paddlepaddle superatedly appart from paddlleocr othe
# or you could just !pip install "paddleocr[all]@git+https://github.com/PaddlePaddle/PaddleOCR.git"
# Make sure the paddle version matches to paddleOCR version
!pip install paddlepaddle==3.2.0

In [ ]:
# Make import superately make sure the version matches
from paddleocr import PaddleOCR

In [5]:
from PIL import Image
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colormaps
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor
import csv

In [6]:
import os
from dotenv import load_dotenv
# Load environment variable from .env
_ = load_dotenv(override=True)

## 1. PaddleOCR Basic

In [ ]:
# Initialize english OCR model
ocr = PaddleOCR(lang='en')

In [ ]:
from IPython.display import display

image_path = 'noisy_img.jpeg'
img = Image.open(image_path)
display(img)

In [9]:
import paddleocr
import paddle

In [10]:
print("paddleocr version: ", paddleocr.__version__)
print("paddle version: ", paddle.__version__)


paddleocr version:  3.3.3
paddle version:  3.2.0


In [ ]:
# Run OCR
result = ocr.predict(image_path)

In [ ]:
page = result[0]
texts = page['rec_texts'] # recognized text strings
scores = page['rec_scores'] # confidence scores for each text line
boxes = page['rec_polys'] # bounding box coordinates

for text, score, box in zip(texts, scores, boxes): # zip -> ties together items with the same position
  # box is a numpy array
  coords = box.astype(int).tolist() # convert to normal list of ints
  print(f"{text:25} | {score:.3f} | {coords}")

In [ ]:
# Preproccess the image
img = page['doc_preprocessor_res']['output_img']

In [ ]:
# display preprocessed image with all items overlayed
img_plot = img.copy()
for text, box in zip(texts, boxes):
  pts = np.array(box, dtype=int)
  cv2.polylines(img_plot, [pts], True, (0, 255, 0), 2)
  x, y = pts[0]
  cv2.putText(img_plot, text, (x, y-5),
              cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

plt.figure(figsize=(10, 16))
plt.imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Aligned Bounding Boxes (processed Image) \n Rcognized text on the top")
plt.show()

## 1.1 Create the PaddleOCR Tool
Turn PaddleOCR to a tool so that can be used with an agent

In [9]:
from langchain.tools import tool
import base64
from langchain_openai import ChatOpenAI
from typing import List, Dict, Any

@tool
def paddle_ocr_read_document(image_path: str) -> List[Dict[str, Any]]:
  """
  Reads an image from the given path and returns extracted text
  with bounding boxes.

  Returns a list of dictionaries, each containing:
  - 'text': the recognized text string
  - 'bbox': bounding box coordinates [x_min, y_min, x_max, y_max]
  - 'confidence': recognition confidence score (if available)
  """
  try:
    result = ocr.predict(image_path) # from paddleOCR
    page = result[0]

    texts = page['rec_texts']
    boxes = page['dt_polys']
    scores = page.get('rec_scores', [None] * len(texts))

    extracted_items = []
    for text, box, score in zip(texts, boxes, scores):
      x_coords = [point[0] for point in box]
      y_coords = [point[1] for point in box]
      bbox = [min(x_coords), min(y_coords), max(x_coords),
              max(y_coords)]

      item = {
          'text': text,
          'bbox': bbox
      }

      if score is not None:
        item['confidance'] = score
      extracted_items.append(item)
    return extracted_items
  except Exception as e:
    return [{'error': f'Error reading image {e}'}]

## 1.2 Create & Run the Agent


In [ ]:
# 1. Define the list of tools
tools = [paddle_ocr_read_document]

# 2. Set up the OpenAI GPT model
if "OPENAI_API_KEY" not in os.environ:
    print("OPENAI_API_KEY environment variable is not set.")
    llm = None
else:
    llm = ChatOpenAI(
        model="gpt-5-mini",
        temperature=1
    )
    print("OpenAI Chat model initialized.")

In [ ]:
from langchain.agents import create_tool_calling_agent
# 3. Create the OpenAI-compatible prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant designed to extract information "+
        "from documents. "
          "You have access to this tool: "
          "Paddle OCR tool to extract raw texts, bounding boxes for "+
          "each text and confidence score from images "
    ),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

# 4. Create a proper tool-calling agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 5. Set up the AgentExecutor to run the tool-enabled loop
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# Run the agent and extract information
task = """
Process the document at 'noisy_img.jpeg' and evaluate that
the total amount is correct.
"""

response = agent_executor.invoke({"input": task})

## 1.3. Run PaddleOCR on the Table and Handwriting examples

In [ ]:
def run_ocr(image_path, ocr_model=ocr, show_text=True):
  display(Image.open(image_path))
  result = ocr.predict(image_path) # result from paddleOCR

  page = result[0]
  texts = page['rec_texts']
  scores = page['rec_scores']
  boxes = page['rec_polys']

  for text, score, box in zip(texts, scores, boxes):
    coords = box.astype(int).tolist()
    print(f'{text:25} | {score:.3f} | {coords}')

  # post proccessed image
  img = page['doc_preprocessor_res']['output_img']
  img_plot = img.copy()

  image_path = Path(image_path)
  output_path = image_path.with_stem(image_path.stem + '_output')
  cv2.imwrite(str(output_path), img, [cv2.IMWRITE_JPEG_QUALITY, 70])

  # Make some annotation on top using bounding box cordinate
  for text, box in zip(texts, boxes):
    pts = np.array(box, dtype=int)
    cv2.polylines(img_plot, [pts], True, (0, 255, 0), 2)
    x, y = pts[0]
    if show_text:
      cv2.putText(img_plot, text,
                  (x, y-5), cv2.FONT_HERSHEY_SIMPLEX,
                  1.0, (255, 0, 0), 2)

  plt.figure(figsize=(10, 12))
  plt.imshow(cv2.cvtColor(img_plot, cv2.COLOR_RGB2RGBA))
  plt.axis('off')

In [ ]:
# image source: https://www.nature.com/articles/s41597-024-03605-5/figures/4
run_ocr('open-ai-rate-limit.png')

In [ ]:
task = """
This is the OpenAI rate limit table. I am only using it for extracting text from images and chat with the extracted text
information from the image. My daily usage is between 5 to 10 hit now please process the document at 'open-ai-rate-limit.png'
and tell me which model is the best I can use to avoid an extra payment.
"""

response = agent_executor.invoke({'input': task})

# Display results side by side
print('\n' + '_'*35 + 'LLM RESULT ' + '_'*33)
print("="*80)
print(response['output'])
print('='*80)

## Extract topics from past exam paper

Create a tool

In [42]:
from langchain.tools import tool
import base64
from langchain_openai import ChatOpenAI
from typing import List, Dict, Any

@tool
def paddle_ocr_read_document_pdf(doc_path: str) -> List[Dict[str, Any]]:
  """
  Reads a document from the given path and returns the extracted text
  # with bounding boxes and confidence score.

  Returns:
    {
        "full_text": str,
        "items": [
            {
                "text": str,
                "bbox": [x_min, y_min, x_max, y_max],
                "confidence": float
            }
        ]
    }
  """
  try:
    result = ocr.predict(doc_path) # from paddleOCR
    page = result[0]

    texts = page['rec_texts']
    boxes = page['dt_polys']
    scores = page.get('rec_scores', [None] * len(texts))

    extracted_items = []
    full_text_lines = []
    for text, box, score in zip(texts, boxes, scores):
      x_coords = [point[0] for point in box]
      y_coords = [point[1] for point in box]
      bbox = [min(x_coords), min(y_coords),
              max(x_coords), max(y_coords)]

      item = {
          'text': text,
          'bbox': bbox,
          "confidence": score
      }

      if score is not None:
        item['confidance'] = score
      extracted_items.append(item)
      full_text_lines.append(text)
    return {
            "full_text": "\n".join(full_text_lines),
            "items": extracted_items
        }
  except Exception as e:
     return { "error": f"Error reading document: {str(e)}" }

In [43]:
# 1. Define the list of tools
tools = [paddle_ocr_read_document_pdf]

# 2. Set up the OpenAI GPT model
if "OPENAI_API_KEY" not in os.environ:
    print("OPENAI_API_KEY environment variable is not set.")
    llm = None
else:
    llm = ChatOpenAI(
        model="gpt-5-mini",
        temperature=1
    )
    print("OpenAI Chat model initialized.")

OpenAI Chat model initialized.


System prompt for the examp paper text extraction agen

In [44]:
from langchain.agents import create_tool_calling_agent
# 3. Create the OpenAI-compatible prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
          You are an expert university professor and exam designer in Signal and Image Processing.

          Your role is to extract EXAM-RELEVANT knowledge from documents.
          You are NOT allowed to summarize or simplify the document.

          You have access to the following tool:
          - Paddle OCR tool: extracts raw text, bounding boxes, and confidence scores from documents.

          CORE RULES:
          1. Always use the Paddle OCR tool when a document path is provided.
          2. After OCR, analyze the extracted text as an exam designer.
          3. Extract EVERYTHING that could appear in the exam(Nothing should missing), including:
            - topics
            - concepts
            - terms
            - keywords
            - algorithms
            - methods
            - properties
            - assumptions
            - formula names
            - theoretical ideas
          4. Concepts may be:
            - explicitly stated
            - embedded in exercises
            - implied in open-ended questions
            - indirectly referenced
          5. NOTHING should be skipped.

          PRIORITY RULE:
          - Concepts from open-ended questions and exercises are MORE important
            because they usually carry higher marks.

          OUTPUT RULES:
          - Output ONE ordered list only
          - Rank items from MOST important → LEAST important
          - Use short, clear concept names
          - Do NOT explain, define, or summarize
          - Do NOT merge concepts unless inseparable
          - Do NOT invent concepts not present or implied

          Your output must allow a student to study top-to-bottom and confidently pass the exam.
          """
    ),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

# 4. Create a proper tool-calling agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 5. Set up the AgentExecutor to run the tool-enabled loop
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

In [47]:
# Run the agent and extract information
task = """
Process the document at path: exam_example_pt1.pdf

Steps to follow:
1. Use the Paddle OCR tool to extract the full text of the document.
2. Treat the document as a collection of exam sample questions and exercises
   for Signal and Image Processing.
3. Extract ALL exam-relevant topics, terms, concepts, methods, and ideas.
4. Rank them from most important to least important based on likelihood
   of appearing in the exam.
5. Output ONLY the ordered list.

Do not include explanations or summaries.
"""

response = agent_executor.invoke({"input": task})

In [48]:
response['output'].split('\n')

['1. Quantization error',
 '2. Aliasing',
 '3. Real ADC',
 '4. Offset (ADC)',
 '5. Gain error',
 '6. Nonlinearity (ADC)',
 '7. Missing code',
 '8. Hamming window',
 '9. Periodic signal',
 '10. Periodicity condition x(t+T0)=x(t)',
 '11. Fundamental period (smallest repetition interval)',
 '12. Cosine signal',
 '13. Amplitude',
 '14. Frequency (Hz)',
 '15. Phase',
 '16. Angular frequency (2πf)',
 '17. Digital frequency',
 '18. Radians (unit)',
 '19. Radians/second (unit)',
 '20. Seconds (time unit)',
 '21. Time-domain waveform plotting',
 '22. Time axis (Time (s))',
 '23. Signal parameters (e.g., x(t)=20 cos(2π(40)t−0.4π))',
 '24. Principal 2π range (e.g., −2π < θ ≤ 2π)',
 '25. Sampling (implied by aliasing)']